In [1]:

# PHASE 4 - TASK 20A
# Connect our analysis notebook to the MySQL database

import os
import pandas as pd
import numpy as np

from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL


# Find the project folder without using
# a computer-specific Windows path.

project_folder = Path.cwd().resolve()

# If the notebook runs from the notebooks folder,
# move one level up to the main project folder.
if project_folder.name == "notebooks":
    project_folder = project_folder.parent


load_dotenv(project_folder / ".env")

db_url = URL.create(
    drivername="mysql+mysqlconnector",
    username=os.getenv("MYSQL_USER"),
    password=os.getenv("MYSQL_PASSWORD"),
    host=os.getenv("MYSQL_HOST"),
    port=int(os.getenv("MYSQL_PORT", "3306")),
    database=os.getenv("MYSQL_DATABASE")
)

engine = create_engine(db_url)

print("Analysis notebook is ready!")


Analysis notebook is ready!


In [2]:

# Load KYC cases and customer records

with engine.connect() as connection:
    df_cases = pd.read_sql(
        text("SELECT * FROM kyc_cases"),
        connection
    )

    df_customers = pd.read_sql(
        text("SELECT * FROM customers"),
        connection
    )

print("KYC cases:", df_cases.shape)
print("Customers:", df_customers.shape)

display(df_cases.head())


KYC cases: (1630, 11)
Customers: (1200, 19)


,case_id,customer_id,review_type,trigger_event,opened_at,due_at,priority,assigned_analyst_id,case_status,complexity,scenario_id
0,KYC000001,CUS00001,ONBOARDING,NEW_TO_BANK,2026-10-05,2026-10-19,HIGH,USR003,ESCALATED,STANDARD,SC01
1,KYC000002,CUS00002,ONBOARDING,NEW_TO_BANK,2026-10-09,2026-10-23,HIGH,USR004,IN_REVIEW,STANDARD,SC02
2,KYC000003,CUS00003,ONBOARDING,NEW_TO_BANK,2026-09-29,2026-10-13,HIGH,USR011,PENDING_SCREENING,STANDARD,SC03
3,KYC000004,CUS00004,ONBOARDING,NEW_TO_BANK,2025-01-12,2025-01-26,HIGH,USR012,COMPLETED,COMPLEX,NaN
4,KYC000005,CUS00005,ONBOARDING,NEW_TO_BANK,2024-02-16,2024-03-01,HIGH,USR008,COMPLETED,COMPLEX,NaN


In [3]:

# TASK 20B: BASIC DATA QUALITY CHECKS

# Check 1: Total records
print("Total KYC cases:", len(df_cases))
print("Total customers:", len(df_customers))

# Check 2: Missing values in KYC cases
print("\nMissing values in KYC cases:")
print(df_cases.isnull().sum())

# Check 3: Duplicate IDs
print("\nDuplicate case IDs:",
      df_cases["case_id"].duplicated().sum())

print("Duplicate customer IDs:",
      df_customers["customer_id"].duplicated().sum())


Total KYC cases: 1630
Total customers: 1200

Missing values in KYC cases:
case_id                   0
customer_id               0
review_type               0
trigger_event             0
opened_at                 0
due_at                    0
priority                  0
assigned_analyst_id       0
case_status               0
complexity                0
scenario_id            1619
dtype: int64

Duplicate case IDs: 0
Duplicate customer IDs: 0


In [4]:

# TASK 20B: DATA QUALITY SUMMARY

print("Duplicate Case IDs:",
      df_cases["case_id"].duplicated().sum())

print("Duplicate Customer IDs:",
      df_customers["customer_id"].duplicated().sum())

print("\nKYC Case Columns With Missing Values:")
print(df_cases.isnull().sum().loc[lambda x: x > 0])

print("\nCustomer Columns With Missing Values:")
print(df_customers.isnull().sum().loc[lambda x: x > 0])


Duplicate Case IDs: 0
Duplicate Customer IDs: 0

KYC Case Columns With Missing Values:
scenario_id    1619
dtype: int64

Customer Columns With Missing Values:
industry                   785
occupation                 415
date_of_birth              415
incorporation_date         785
registration_number        785
initial_onboarding_date    244
dtype: int64


In [5]:

# TASK 20C: KYC CASE STATUS ANALYSIS

# Count how many cases belong to each status
status_summary = (
    df_cases["case_status"]
    .value_counts()
    .reset_index()
)

# Give the output columns meaningful names
status_summary.columns = ["case_status", "total_cases"]

# Calculate percentage of total cases
status_summary["percentage"] = (
    status_summary["total_cases"] / len(df_cases) * 100
).round(2)

# Display the result
display(status_summary)


,case_status,total_cases,percentage
0,COMPLETED,1259,77.24
1,IN_REVIEW,88,5.40
2,PENDING_INFORMATION,83,5.09
3,ESCALATED,70,4.29
4,PENDING_SCREENING,45,2.76
5,QC_REWORK,37,2.27
6,AWAITING_QC,31,1.90
7,DECLINED,17,1.04


In [6]:

# TASK 20C: ACTIVE CASE PRIORITY ANALYSIS

# Define statuses that represent active work
active_statuses = [
    "IN_REVIEW",
    "PENDING_INFORMATION",
    "ESCALATED",
    "PENDING_SCREENING",
    "QC_REWORK",
    "AWAITING_QC"
]

# Filter only active KYC cases
df_active = df_cases[
    df_cases["case_status"].isin(active_statuses)
].copy()

# Count cases by priority
priority_summary = (
    df_active["priority"]
    .value_counts()
    .reset_index()
)

priority_summary.columns = ["priority", "total_cases"]

# Show the results
print("Total active KYC cases:", len(df_active))
display(priority_summary)


Total active KYC cases: 354


,priority,total_cases
0,MEDIUM,132
1,LOW,118
2,HIGH,104


In [7]:

# TASK 20C: OVERDUE HIGH-PRIORITY KYC CASES

# Convert the deadline column into datetime
df_active["due_at"] = pd.to_datetime(
    df_active["due_at"]
)

# Use the same snapshot date as Phase 2
snapshot_date = pd.Timestamp("2026-11-05")

# Calculate days overdue
df_active["days_overdue"] = (
    snapshot_date - df_active["due_at"].dt.normalize()
).dt.days

# Filter cases requiring attention
overdue_high = df_active[
    (df_active["priority"] == "HIGH") &
    (df_active["case_status"] == "PENDING_INFORMATION") &
    (df_active["days_overdue"] > 0)
].copy()

# Display investigation results
print("Overdue HIGH-priority cases:", len(overdue_high))

display(
    overdue_high[
        ["case_id", "customer_id", "priority",
         "case_status", "due_at", "days_overdue"]
    ].sort_values("days_overdue", ascending=False).head(10)
)


Overdue HIGH-priority cases: 16


,case_id,customer_id,priority,case_status,due_at,days_overdue
1371,KYC001372,CUS00420,HIGH,PENDING_INFORMATION,2026-09-15,51
1414,KYC001415,CUS00114,HIGH,PENDING_INFORMATION,2026-09-17,49
1565,KYC001566,CUS00453,HIGH,PENDING_INFORMATION,2026-10-03,33
1122,KYC001123,CUS01123,HIGH,PENDING_INFORMATION,2026-10-04,32
1067,KYC001068,CUS01068,HIGH,PENDING_INFORMATION,2026-10-11,25
1055,KYC001056,CUS01056,HIGH,PENDING_INFORMATION,2026-10-13,23
1113,KYC001114,CUS01114,HIGH,PENDING_INFORMATION,2026-10-20,16
1021,KYC001022,CUS01022,HIGH,PENDING_INFORMATION,2026-10-21,15
1412,KYC001413,CUS00466,HIGH,PENDING_INFORMATION,2026-10-22,14
984,KYC000985,CUS00985,HIGH,PENDING_INFORMATION,2026-10-23,13


In [8]:

# TASK 20D: INVESTIGATE A SPECIFIC KYC CASE

# Select the case we want to investigate
case_id = "KYC001203"

# Find this case in our active KYC dataset
case_record = df_active.loc[
    df_active["case_id"] == case_id
]

# Display important case information
display(
    case_record[
        [
            "case_id",
            "customer_id",
            "review_type",
            "case_status",
            "priority",
            "due_at",
            "days_overdue"
        ]
    ]
)


,case_id,customer_id,review_type,case_status,priority,due_at,days_overdue
1202,KYC001203,CUS00006,PERIODIC,PENDING_INFORMATION,HIGH,2026-10-29,7


In [9]:

# TASK 20D: RETRIEVE CUSTOMER PROFILE

# Get the customer ID from the selected KYC case
customer_id = case_record["customer_id"].iloc[0]

# Find the matching customer record
customer_record = df_customers.loc[
    df_customers["customer_id"] == customer_id
]

# Display the customer details vertically
display(customer_record.T)


,5
customer_id,CUS00006
customer_type,INDIVIDUAL
legal_name,Alice Bennett
country_code,IE
nationality_code,IE
industry,NaN
occupation,Business Owner
date_of_birth,1970-04-21
incorporation_date,None
registration_number,NaN


In [10]:

# TASK 20D: FIND DOCUMENT AND INFORMATION REQUEST TABLES

query = text("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = DATABASE()
      AND (
          table_name LIKE '%document%'
          OR table_name LIKE '%request%'
      )
    ORDER BY table_name
""")

with engine.connect() as connection:
    related_tables = pd.read_sql(query, connection)

display(related_tables)


,TABLE_NAME
0,customer_documents
1,information_requests


In [11]:

# TASK 20D: INSPECT DOCUMENT AND REQUEST TABLE STRUCTURE

query = text("""
    SELECT
        table_name,
        column_name,
        data_type
    FROM information_schema.columns
    WHERE table_schema = DATABASE()
      AND table_name IN (
          'customer_documents',
          'information_requests'
      )
    ORDER BY table_name, ordinal_position
""")

with engine.connect() as connection:
    table_structure = pd.read_sql(query, connection)

display(table_structure)


,TABLE_NAME,COLUMN_NAME,DATA_TYPE
0,customer_documents,document_id,varchar
1,customer_documents,case_id,varchar
2,customer_documents,customer_id,varchar
3,customer_documents,subject_type,varchar
4,customer_documents,subject_id,varchar
5,customer_documents,document_type,varchar
6,customer_documents,document_reference,varchar
7,customer_documents,issued_at,date
8,customer_documents,expires_at,date
9,customer_documents,received_at,date


In [12]:

# TASK 20D: REVIEW ALICE BENNETT'S KYC DOCUMENTS

# Retrieve documents linked to the selected case
query = text("""
    SELECT
        document_id,
        document_type,
        verification_status,
        exception_code,
        issued_at,
        expires_at,
        received_at,
        verified_at
    FROM customer_documents
    WHERE case_id = :case_id
      AND customer_id = :customer_id
    ORDER BY document_type
""")

# Execute the SQL query and load results into pandas
with engine.connect() as connection:
    df_documents = pd.read_sql(
        query,
        connection,
        params={
            "case_id": case_id,
            "customer_id": customer_id
        }
    )

# Display Alice's KYC documents
display(df_documents)


,document_id,document_type,verification_status,exception_code,issued_at,expires_at,received_at,verified_at
0,DOC0003242,ADDRESS_PROOF,VERIFIED,NaN,2024-07-15,2029-02-09,2026-10-09,2026-10-10
1,DOC0003241,IDENTITY_PROOF,EXPIRED,EXPIRED_AT_REVIEW,2024-09-04,2026-10-03,2026-10-09,None
2,DOC0003243,SOURCE_OF_FUNDS_EVIDENCE,VERIFIED,NaN,2024-05-13,None,2026-10-09,2026-10-10


In [13]:

# TASK 20D: CHECK ALICE'S INFORMATION REQUEST

query = text("""
    SELECT
        request_id,
        requested_item,
        request_status,
        requested_at,
        due_at,
        responded_at,
        relationship_manager_id
    FROM information_requests
    WHERE case_id = :case_id
""")

with engine.connect() as connection:
    df_requests = pd.read_sql(
        query,
        connection,
        params={"case_id": "KYC001203"}
    )

display(df_requests)


,request_id,requested_item,request_status,requested_at,due_at,responded_at,relationship_manager_id
0,REQ000072,REFRESH_IDENTITY,OPEN,2026-10-10,2026-10-17,None,USR020


### KYC Case Review – Alice Bennett

**Case ID:** KYC001203  
**Customer ID:** CUS00006  
**Review Type:** Periodic KYC Review  
**Priority:** High  
**Case Status:** Pending Information  
**Review Snapshot:** 05 November 2026

#### Review Findings

I reviewed the customer's KYC profile, supporting documents and outstanding information request to understand why the periodic review is still pending.

The address proof and source-of-funds document have both been verified. However, the identity document (DOC0003241) expired on 03 October 2026 and was received on 09 October 2026, after its expiry date. The document is marked as EXPIRED, with the exception code `EXPIRED_AT_REVIEW`.

An information request (REQ000072) was raised on 10 October 2026 to obtain an updated identity document. The request is still marked as OPEN and had a due date of 17 October 2026.

As of the review snapshot, the information request is 19 calendar days past its due date. The KYC case itself was due on 29 October 2026 and is 7 calendar days overdue.

#### Assessment

The main reason this case remains pending is the expired identity document. The other two documents reviewed have been verified, but the outstanding identity-document requirement still needs to be resolved before the periodic review can move forward.

The case is marked as high priority, and both the information-request deadline and case deadline have passed. Follow-up is therefore required.

#### Recommended Next Action

Follow up with the assigned Relationship Manager (USR020) to check the status of the replacement identity document. Once received, the document should be reviewed and verified according to the bank's KYC requirements.

The case should remain pending until the outstanding requirement is resolved. Given the overdue status and high priority, the need for escalation should also be checked against the bank's internal procedures.

**Current Recommendation:** Continue follow-up on the outstanding identity document and reassess the case once updated evidence is available.

In [14]:

# TASK 21C: FIND SCREENING AND RISK TABLES

query = text("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = DATABASE()
      AND (
          table_name LIKE '%screen%'
          OR table_name LIKE '%risk%'
          OR table_name LIKE '%watchlist%'
      )
    ORDER BY table_name
""")

with engine.connect() as connection:
    tables = pd.read_sql(query, connection)

display(tables)


,TABLE_NAME
0,case_risk_factors
1,customer_risk_assessments
2,screening_hits
3,screening_runs
4,screening_sources
5,watchlist_aliases
6,watchlist_entities
7,watchlist_relationships


In [15]:

# TASK 21C: CHECK SCREENING TABLE COLUMNS

query = text("""
    SELECT
        table_name,
        column_name,
        data_type
    FROM information_schema.columns
    WHERE table_schema = DATABASE()
      AND table_name IN (
          'screening_runs',
          'screening_hits'
      )
    ORDER BY table_name, ordinal_position
""")

with engine.connect() as connection:
    screening_columns = pd.read_sql(query, connection)

display(screening_columns)


,TABLE_NAME,COLUMN_NAME,DATA_TYPE
0,screening_hits,screening_hit_id,varchar
1,screening_hits,screening_run_id,varchar
2,screening_hits,watchlist_entity_id,varchar
3,screening_hits,match_score,decimal
4,screening_hits,match_basis,varchar
5,screening_hits,identity_assessment,varchar
6,screening_hits,analyst_disposition,varchar
7,screening_hits,reviewed_at,date
8,screening_hits,reviewed_by,varchar
9,screening_hits,analyst_rationale,varchar


In [16]:

# TASK 21C: CHECK RISK ASSESSMENT TABLE COLUMNS

query = text("""
    SELECT
        table_name,
        column_name,
        data_type
    FROM information_schema.columns
    WHERE table_schema = DATABASE()
      AND table_name IN (
          'customer_risk_assessments',
          'case_risk_factors'
      )
    ORDER BY table_name, ordinal_position
""")

with engine.connect() as connection:
    risk_columns = pd.read_sql(query, connection)

display(risk_columns)


,TABLE_NAME,COLUMN_NAME,DATA_TYPE
0,case_risk_factors,factor_id,varchar
1,case_risk_factors,case_id,varchar
2,case_risk_factors,factor_name,varchar
3,case_risk_factors,factor_value,decimal
4,case_risk_factors,weight,decimal
5,case_risk_factors,weighted_value,decimal
6,case_risk_factors,evidence_note,varchar
7,customer_risk_assessments,assessment_id,varchar
8,customer_risk_assessments,case_id,varchar
9,customer_risk_assessments,assessment_score,decimal


In [17]:

# TASK 21D: FIND INVESTIGATION AND AUDIT TABLES

query = text("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = DATABASE()
      AND (
          table_name LIKE '%note%'
          OR table_name LIKE '%action%'
          OR table_name LIKE '%audit%'
          OR table_name LIKE '%history%'
      )
    ORDER BY table_name
""")

with engine.connect() as connection:
    result = pd.read_sql(query, connection)

display(result)


,TABLE_NAME
0,case_audit_log
1,case_notes


In [18]:

# TASK 21D: CHECK CASE NOTES AND AUDIT LOG COLUMNS

query = text("""
    SELECT
        table_name,
        column_name,
        data_type,
        is_nullable,
        column_key
    FROM information_schema.columns
    WHERE table_schema = DATABASE()
      AND table_name IN (
          'case_notes',
          'case_audit_log'
      )
    ORDER BY table_name, ordinal_position
""")

with engine.connect() as connection:
    table_columns = pd.read_sql(query, connection)

display(table_columns)


,TABLE_NAME,COLUMN_NAME,DATA_TYPE,IS_NULLABLE,COLUMN_KEY
0,case_audit_log,event_id,varchar,NO,PRI
1,case_audit_log,case_id,varchar,YES,MUL
2,case_audit_log,event_at,date,YES,
3,case_audit_log,actor_id,varchar,YES,MUL
4,case_audit_log,event_type,varchar,YES,
5,case_audit_log,related_table,varchar,YES,
6,case_audit_log,related_record_id,varchar,YES,
7,case_audit_log,old_value,varchar,YES,
8,case_audit_log,new_value,varchar,YES,
9,case_audit_log,event_comment,varchar,YES,


In [19]:

# TASK 21D: CHECK EXISTING NOTE AND AUDIT IDS

with engine.connect() as connection:

    notes = pd.read_sql(
        text("""
            SELECT note_id, note_type, author_id
            FROM case_notes
            ORDER BY note_id DESC
            LIMIT 5
        """),
        connection
    )

    audit = pd.read_sql(
        text("""
            SELECT event_id, event_type, actor_id
            FROM case_audit_log
            ORDER BY event_id DESC
            LIMIT 5
        """),
        connection
    )

    field_lengths = pd.read_sql(
        text("""
            SELECT
                table_name,
                column_name,
                character_maximum_length
            FROM information_schema.columns
            WHERE table_schema = DATABASE()
              AND table_name IN ('case_notes', 'case_audit_log')
              AND column_name IN (
                  'note_id', 'event_id', 'note_text'
              )
        """),
        connection
    )

print("EXISTING NOTE IDS")
display(notes)

print("EXISTING AUDIT EVENT IDS")
display(audit)

print("FIELD LENGTHS")
display(field_lengths)


EXISTING NOTE IDS


,note_id,note_type,author_id
0,NOTE0003243,DECISION,USR010
1,NOTE0003242,INTAKE,USR010
2,NOTE0003241,DECISION,USR010
3,NOTE0003240,INTAKE,USR010
4,NOTE0003239,INVESTIGATION,USR006


EXISTING AUDIT EVENT IDS


,event_id,event_type,actor_id
0,EVT0006903,CASE_CLOSED,USR010
1,EVT0006902,QC_REVIEW,USR015
2,EVT0006901,SCREENING_RUN,USR010
3,EVT0006900,CASE_CREATED,USR010
4,EVT0006899,CASE_CLOSED,USR010


FIELD LENGTHS


,TABLE_NAME,COLUMN_NAME,CHARACTER_MAXIMUM_LENGTH
0,case_audit_log,event_id,255
1,case_notes,note_id,255
2,case_notes,note_text,1000


In [20]:

# TASK 21D: CHECK INFORMATION REQUEST STATUSES

query = text("""
    SELECT
        request_status,
        COUNT(*) AS total_requests
    FROM information_requests
    GROUP BY request_status
    ORDER BY total_requests DESC
""")

with engine.connect() as connection:
    status_df = pd.read_sql(query, connection)

display(status_df)


,request_status,total_requests
0,OPEN,121


In [21]:

# TASK 21D: CHECK INFORMATION REQUEST TABLE RULES

with engine.connect() as connection:
    result = connection.execute(
        text("SHOW CREATE TABLE information_requests")
    )

    table_definition = result.fetchone()[1]

print(table_definition)


CREATE TABLE `information_requests` (
  `request_id` varchar(255) NOT NULL,
  `case_id` varchar(255) DEFAULT NULL,
  `relationship_manager_id` varchar(255) DEFAULT NULL,
  `requested_item` varchar(255) DEFAULT NULL,
  `requested_at` date DEFAULT NULL,
  `due_at` date DEFAULT NULL,
  `responded_at` date DEFAULT NULL,
  `request_status` varchar(255) DEFAULT NULL,
  `request_detail` varchar(1000) DEFAULT NULL,
  `closure_note` varchar(1000) DEFAULT NULL,
  PRIMARY KEY (`request_id`),
  KEY `fk_information_requests_relationship_manager_id` (`relationship_manager_id`),
  KEY `idx_information_requests_case_id` (`case_id`),
  CONSTRAINT `fk_information_requests_case_id` FOREIGN KEY (`case_id`) REFERENCES `kyc_cases` (`case_id`),
  CONSTRAINT `fk_information_requests_relationship_manager_id` FOREIGN KEY (`relationship_manager_id`) REFERENCES `users` (`user_id`)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_0900_ai_ci


In [22]:

# TASK 21D: FIND A CASE FOR UPDATE TESTING

query = text("""
    SELECT
        k.case_id,
        k.customer_id,
        k.case_status,
        k.priority
    FROM kyc_cases AS k
    WHERE k.case_status = 'IN_REVIEW'
      AND NOT EXISTS (
          SELECT 1
          FROM information_requests AS ir
          WHERE ir.case_id = k.case_id
      )
    ORDER BY k.case_id
    LIMIT 5
""")

with engine.connect() as connection:
    practice_cases = pd.read_sql(query, connection)

display(practice_cases)


,case_id,customer_id,case_status,priority
0,KYC000002,CUS00002,IN_REVIEW,HIGH
1,KYC000968,CUS00968,IN_REVIEW,MEDIUM
2,KYC000971,CUS00971,IN_REVIEW,LOW
3,KYC000975,CUS00975,IN_REVIEW,LOW
4,KYC000977,CUS00977,IN_REVIEW,HIGH


In [23]:

# TASK 21D: CREATE A PRACTICE INFORMATION REQUEST

from uuid import uuid4

# Generate a unique ID for our test request
request_id = "REQ_TEST_" + uuid4().hex[:12].upper()

# Insert a clearly labelled practice request
query = text("""
    INSERT INTO information_requests
    (
        request_id,
        case_id,
        requested_item,
        requested_at,
        due_at,
        request_status,
        request_detail
    )
    VALUES
    (
        :request_id,
        :case_id,
        'TRAINING_TEST_ONLY',
        CURDATE(),
        DATE_ADD(CURDATE(), INTERVAL 7 DAY),
        'OPEN',
        'Temporary practice request for testing the app'
    )
""")

with engine.begin() as connection:
    connection.execute(
        query,
        {
            "request_id": request_id,
            "case_id": "KYC000971"
        }
    )

print("Practice request created:", request_id)
print("Initial status: OPEN")


Practice request created: REQ_TEST_43DE0DE4309E
Initial status: OPEN


In [24]:

# TASK 21D: CLEAN UP PRACTICE REQUEST

request_id = "REQ_TEST_43DE0DE4309E"
event_id = "EVT_DF73CF5C645E4DBABF990C2C026F6A5B"

with engine.begin() as connection:

    # Delete the audit record created during our test
    audit_result = connection.execute(
        text("""
            DELETE FROM case_audit_log
            WHERE event_id = :event_id
              AND related_record_id = :request_id
              AND event_type = 'REQUEST_STATUS_CHANGED'
        """),
        {"event_id": event_id, "request_id": request_id}
    )

    # Delete only our labelled practice request
    request_result = connection.execute(
        text("""
            DELETE FROM information_requests
            WHERE request_id = :request_id
              AND case_id = 'KYC000971'
              AND requested_item = 'TRAINING_TEST_ONLY'
        """),
        {"request_id": request_id}
    )

    # Ensure exactly one of each was deleted
    if audit_result.rowcount != 1 or request_result.rowcount != 1:
        raise ValueError("Cleanup verification failed.")

print("Practice request and audit event removed successfully!")


Practice request and audit event removed successfully!
